In [ ]:
!pip install transformers datasets evaluate torch scikit-learn gradio

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ag_news")

print(dataset)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01
)

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    acc = accuracy.compute(
        predictions=predictions,
        references=labels
    )

    f1_score = f1.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": acc["accuracy"],
        "f1": f1_score["f1"]
    }

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
results = trainer.evaluate()

print(results)

In [ ]:
model.save_pretrained("saved_model")

tokenizer.save_pretrained(
    "saved_model"
)

In [ ]:
import gradio as gr
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="saved_model"
)

labels = {
    "LABEL_0":"World",
    "LABEL_1":"Sports",
    "LABEL_2":"Business",
    "LABEL_3":"Sci/Tech"
}

def predict_news(text):

    result = classifier(text)[0]

    return labels[result["label"]]

demo = gr.Interface(
    fn=predict_news,
    inputs="text",
    outputs="text",
    title="News Topic Classifier"
)

demo.launch()